# 12 · Facility siting and equity tradeoffs

**Question:** Does the best average travel time also provide the fairest access?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Give Astra the candidate table and an explicit objective, require a decision with numerical evidence, then verify it by exhaustive enumeration.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Choose an objective · 30 minutes
**Learn:** separate efficiency from coverage, examine geographic groups, and audit an optimization recommendation.
The invented city has 1,200 residents. West/east groups describe location only, not ethnicity,
income or disease status. A barrier has one northern crossing; all edges are bidirectional.
Choose one additional facility. We compare minimum population-weighted mean time with maximum
worst-group coverage within `threshold_min`. Neither objective represents service capacity or quality.


In [ ]:
from experiments import access_summary, grade_answer
c=city(); pop=allocate(c['capacity'],1200)
nodes,graph=network(blocked=[(r*11+5,r*11+6) for r in range(10)])
idx,connector=snap(c['xy'],nodes)
groups={'west':c['xy'][:,0]<1100,'east':c['xy'][:,0]>=1100}
clinic=22; speed_m_min=80.; threshold_min=15.
candidate_nodes=[6,28,50,72,94,116]
rows=[]; candidate_times={}
for candidate in candidate_nodes:
    minutes=(shortest(graph,[clinic,candidate])[idx]+connector)/speed_m_min
    candidate_times[candidate]=minutes
    overall=access_summary(minutes,pop,threshold_min)
    coverage={name:access_summary(minutes[mask],pop[mask],threshold_min)['coverage'] for name,mask in groups.items()}
    rows.append(dict(node=candidate,mean_min=overall['mean_reachable_min'],coverage=overall['coverage'],
                     worst_group_coverage=min(coverage.values()),group_coverage=coverage,
                     unreachable_population=overall['unreachable_population']))
efficiency=min(rows,key=lambda r:(r['mean_min'],r['node']))
equity=min(rows,key=lambda r:(-r['worst_group_coverage'],r['mean_min'],r['node']))
assert all(r['unreachable_population']==0 for r in rows)
assert all(0<=r['worst_group_coverage']<=r['coverage']<=1 for r in rows)
print('node | mean minutes | west coverage | east coverage')
for r in rows: print(r['node'],round(r['mean_min'],3),{k:round(v,3) for k,v in r['group_coverage'].items()})
print('Efficiency choice:',efficiency['node'],'| equity choice:',equity['node'])


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for u,edges in graph.items():
    for v,w in edges:
        if u<v: axes[0].plot(*nodes[[u,v]].T,color='#bbc5cc',lw=.6)
for name,mask in groups.items(): axes[0].scatter(*c['xy'][mask].T,s=15,label=name)
axes[0].scatter(*nodes[clinic],marker='P',s=130,c='black',label='Existing')
for node in candidate_nodes:
    axes[0].scatter(*nodes[node],marker='*',s=130,c='#b34f00'); axes[0].annotate(str(node),nodes[node])
map_axes(axes[0],'Synthetic geographic groups'); axes[0].legend()
axes[1].scatter([r['mean_min'] for r in rows],[r['worst_group_coverage'] for r in rows],s=90)
for r in rows: axes[1].annotate(str(r['node']),(r['mean_min'],r['worst_group_coverage']),xytext=(5,5),textcoords='offset points')
axes[1].set(xlabel='Population-weighted mean minutes (lower is better)',ylabel='Worst-group coverage (higher is better)',title='Every candidate, two objectives',ylim=(0,1.05))
plt.tight_layout(); plt.show()


## Ask Astra to make a bounded decision
Send only `12_prompt.json`. Its candidate table is evidence, not a preselected answer.
The objective and tie-breaks are declared before evaluation. A model should not substitute a different objective.
Discuss whether group boundaries or travel thresholds change the recommendation; that is a policy choice,
not something the model can resolve from coordinates alone.


In [ ]:
prompt=dict(task='Choose exactly one candidate maximizing worst_group_coverage; break ties by lowest mean_min, then lowest node ID. Return ONLY JSON with chosen_node and worst_group_coverage.',
            synthetic=True,threshold_min=threshold_min,coverage_units='fraction of all group residents',
            candidates=rows,assumptions=['No capacity constraint','Local synthetic groups','All residents reachable'])
export_json('12_prompt.json',prompt)
expected=dict(chosen_node=equity['node'],worst_group_coverage=equity['worst_group_coverage'])
response_source='FIXTURE - efficiency objective substituted for equity'
model_name='not run'; trial_date='not run'; interface='fixture'
response_text=json.dumps(dict(chosen_node=efficiency['node'],worst_group_coverage=efficiency['worst_group_coverage']))
score=grade_answer(response_text,expected,{'worst_group_coverage':.001})
display(score)
export_json('12_trial.json',dict(response_source=response_source,model=model_name,date=trial_date,
            interface=interface,prompt=prompt,response_text=response_text,score=score,reference=expected,
            coverage_tolerance=.001))


## Check a misleading denominator
A disconnected person must count as unserved. Dropping disconnected residents from both the numerator
and denominator can make a damaged network appear to perform better.
**Try:** set the service threshold to 10 or 20 minutes. Do the efficiency and equity choices still differ?
Agreement for some parameters is a useful result too; do not force a tradeoff.


In [ ]:
tiny=access_summary([5.,float('inf')],[60,40],15.)
assert tiny['coverage']==.6 and tiny['unreachable_population']==40
assert tiny['mean_reachable_min']==5.
print('60 served + 40 disconnected: coverage =',tiny['coverage'],'not 100%.')


## Extended investigation: Assign residents to capacity-limited facilities

For each candidate clinic, assign residents using a transparent shortest-pair greedy rule with an explicitly unserved category. Compare 200, 400 and 600 places of added capacity. Report population conservation, facility utilization and geographic-group service fractions. Candidate enumeration is exhaustive, but the assignment inside each candidate is a heuristic; its result must not be labeled a globally optimal allocation.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Assign residents to capacity-limited facilities', ['Demand + candidate travel times', 'Capacity-constrained heuristic', 'Flows / unserved / group coverage', 'Service capacity is an assumption'])
plt.show()


In [ ]:
from extension_tools import capacity_assignment
ex_c=city(); ex_pop=allocate(ex_c['capacity'],1200); ex_nodes,ex_graph=network(blocked=[(r*11+5,r*11+6) for r in range(10)])
ex_idx,ex_connector=snap(ex_c['xy'],ex_nodes); ex_west=ex_c['xy'][:,0]<1100; ex_results=[]
for added in [200,400,600]:
    for candidate in [6,28,50,72,94,116]:
        times=np.column_stack([(shortest(ex_graph,[node])[ex_idx]+ex_connector)/80 for node in [22,candidate]])
        # A 20-minute service limit is part of eligibility, not just presentation.
        times[times>20]=np.inf; assignment,unserved=capacity_assignment(times,ex_pop,[600,added])
        group_coverage=[float(assignment[mask].sum()/ex_pop[mask].sum()) for mask in [ex_west,~ex_west]]
        ex_results.append(dict(added_capacity=added,node=candidate,served=int(assignment.sum()),unserved=int(unserved.sum()),worst_group=min(group_coverage),utilization=assignment.sum(axis=0).tolist()))
        assert (assignment.sum(axis=0)<=np.array([600,added])).all()
ex_choice=min([r for r in ex_results if r['added_capacity']==400],key=lambda r:(-r['worst_group'],-r['served'],r['node']))
times=np.column_stack([(shortest(ex_graph,[node])[ex_idx]+ex_connector)/80 for node in [22,ex_choice['node']]])
times[times>20]=np.inf; ex_assignment,ex_unserved=capacity_assignment(times,ex_pop,[600,400])
fig,axes=plt.subplots(1,3,figsize=(14,4))
for added in [200,400,600]:
    subset=[r for r in ex_results if r['added_capacity']==added]; axes[0].plot([r['node'] for r in subset],[r['worst_group'] for r in subset],marker='o',label=f'+{added} places')
axes[0].set(xlabel='Candidate node',ylabel='Worst-group served fraction',title='Capacity changes the siting objective'); axes[0].legend(fontsize=8)
for j,node in enumerate([22,ex_choice['node']]):
    for i in np.flatnonzero(ex_assignment[:,j]): axes[1].plot([ex_c['xy'][i,0],ex_nodes[node,0]],[ex_c['xy'][i,1],ex_nodes[node,1]],alpha=.2,lw=.5,c=['#087f8c','#b87922'][j])
axes[1].scatter(*ex_nodes[[22,ex_choice['node']]].T,c='black',marker='*',s=100); map_axes(axes[1],'Assignment desire lines (not street routes)')
dots=axes[2].scatter(*ex_c['xy'].T,c=ex_unserved,cmap='magma',s=30); map_axes(axes[2],'Explicit unserved residents'); fig.colorbar(dots,ax=axes[2],label='People')
fig.tight_layout(); extension_figure=fig
extension_evidence=dict(candidates=[r for r in ex_results if r['added_capacity']==400],objective='maximize worst_group, then served, then lowest node',total_population=1200,service_limit_min=20)
extension_expected=dict(chosen_node=ex_choice['node'],unserved=ex_choice['unserved'])
extension_task='Select from the candidate results under the declared objective. Return chosen_node and unserved. These are results of a capacity-limited heuristic, not a claim of global optimality.'


In [ ]:
extension_prompt,extension_reference=export_investigation('12',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 12_extension_prompt.json and optionally 12_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `12_extension_prompt.json` and `12_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('12_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
